In [3]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

# Validación y carga
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Modelos desde el .env
GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4o-mini")
FAST_MODEL = os.getenv("OPENAI_FAST_MODEL", "gpt-4o-mini")

print("Directorio de trabajo actual:", Path.cwd())
print("Modelo generativo:", GENERATION_MODEL)
print("Modelo rápido:", FAST_MODEL)
print("¡Entorno configurado correctamente!")

Directorio de trabajo actual: c:\Users\Manuel\Documents\PONTIA\6-Large Lenguage Models\Evaluacion_Manuel
Modelo generativo: gpt-4.1-mini
Modelo rápido: gpt-4.1-mini
¡Entorno configurado correctamente!


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# PDF
loader = PyPDFLoader("data/TENERIFE.pdf")
docs = loader.load()

# Chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150
)
chunks = text_splitter.split_documents(docs)

# Embeddings y FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"¡PDF cargado con éxito! Se han indexado {len(chunks)} fragmentos en FAISS.")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************MLQA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
# Contenido del primer chunk y metadatos
print("--- EJEMPLO DEL PRIMER CHUNK ---")
print(f"Metadatos: {chunks[0].metadata}")
print(f"Contenido:\n{chunks[0].page_content}\n")
print("-" * 50)

pregunta_prueba = "¿Qué puedo visitar en Santa Cruz de Tenerife?"
print(f"\n--- PRUEBA DEL RETRIEVER ---")
print(f"Buscando información para: '{pregunta_prueba}'\n")

#Consulta a FAISS
documentos_recuperados = retriever.invoke(pregunta_prueba)

# Mejor resultado
if documentos_recuperados:
    print("Mejor fragmento encontrado:")
    print(documentos_recuperados[0].page_content)
    print(f"\nFuente: Página {documentos_recuperados[0].metadata.get('page', 'Desconocida')}")
else:
    print("No se encontró información relevante.")

--- EJEMPLO DEL PRIMER CHUNK ---
Metadatos: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': 'data/TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1'}
Contenido:
TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - 
ubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del 
Príncipe - ubicación). 
- Volver de nuevo a 

In [ ]:
import logging
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Esquema de entrada
class WeatherInput(BaseModel):
    fecha: str = Field(description="La fecha para la cual se solicita el tiempo, en formato YYYY-MM-DD o lenguaje natural (ej. 'hoy', 'mañana').")
    ubicacion: str = Field(default="Tenerife", description="La ubicación para la cual se busca el clima.")

@tool("get_weather", args_schema=WeatherInput, return_direct=False)
def get_weather(fecha: str, ubicacion: str = "Tenerife") -> str:
    """Obtiene la predicción del tiempo para una fecha específica en Tenerife."""
    logger.info(f"Intento de llamada a get_weather para la fecha: '{fecha}' en '{ubicacion}'")
    
    try:
        if not fecha or fecha.strip() == "":
            raise ValueError("La fecha proporcionada no es válida o está vacía.")
            
        # Simular respuesta
        clima_simulado = "soleado con una temperatura media de 24°C"
        logger.info(f"Llamada exitosa. Clima devuelto: {clima_simulado}")
        return f"El tiempo estimado en {ubicacion} para el {fecha} es {clima_simulado}."
        
    except Exception as e:
        logger.error(f"Error al obtener el clima: {str(e)}")
        return f"Lo siento, hubo un error al consultar el servicio meteorológico: {str(e)}"

print("Herramienta 'get_weather' definida correctamente con Pydantic y logs.")

Herramienta 'get_weather' definida correctamente con Pydantic y logs.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools.retriever import create_retriever_tool

# RAG
rag_tool = create_retriever_tool(
    retriever=retriever,
    name="buscar_guia_tenerife",
    description="Usa esta herramienta SIEMPRE para buscar información turística, lugares o datos sobre Tenerife. Debes devolver respuestas citando la fuente."
)

# Herramientas para el agente
tools = [get_weather, rag_tool]

llm = ChatOpenAI(
    model=GENERATION_MODEL, 
    temperature=0.3,
    max_tokens=600
)

# Prompt del sistema
system_prompt = (
    "Eres un asistente turístico experto en Tenerife. "
    "Tu objetivo es ayudar a los turistas usando la herramienta 'buscar_guia_tenerife'. "
    "SIEMPRE debes citar la fuente de la información basándote en la guía. "
    "Si te preguntan por el clima, usa la herramienta 'get_weather'."
)

# Agente
agent_graph = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agente conversacional configurado correctamente.")

Agente conversacional configurado correctamente con la arquitectura moderna.


In [ ]:
# Pruebas de RAG, Function Calling y Mantenimiento de Contexto
pruebas = [
    "Dime qué ruta me recomiendas hacer si visito Santa Cruz de Tenerife.",
    "¿Qué tiempo hará allí el 2026-06-20?",
    "Genial. Siguiendo con la ruta que me dijiste antes, ¿dónde sugieres que aparque al principio?"
]

# Memoria para inyectar en el agente en cada turno
historial = []

for i, pregunta in enumerate(pruebas, 1):
    print(f"\n--- Turno {i} ---")
    print(f"Usuario: {pregunta}")
    
    historial.append({"role": "user", "content": pregunta})
    
    try:
        resultado = agent_graph.invoke({"messages": historial})
        
        respuesta_texto = resultado["messages"][-1].content
        print(f"Asistente: {respuesta_texto}")
        
        # Guardar respuesta
        historial.append({"role": "assistant", "content": respuesta_texto})
        
    except Exception as e:
        print(f"Error en la ejecución: {e}")

print("\n" + "="*50 + "\nPruebas finalizadas. Revisa las citaciones del RAG y el log del clima.")


--- Turno 1 ---
Usuario: Dime qué ruta me recomiendas hacer si visito Santa Cruz de Tenerife.


2026-06-16 20:49:19,365 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 20:49:19,697 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 20:49:22,755 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Asistente: Te recomiendo esta ruta para visitar Santa Cruz de Tenerife:

1. Aparca en el aparcamiento del Parque Marítimo César Manrique, que es un conjunto de piscinas naturales.
2. Camina por la Avenida Marítima hasta llegar a la Plaza de España, pasando por el Auditorio de Tenerife.
3. Desde Plaza de España, callejea un poco subiendo por la Calle Castillo en dirección a Plaza Weyler.
4. Visita el Parque García Sanabria y luego baja hacia Plaza de España pasando por la Plaza del Príncipe.
5. Vuelve a por el coche y dirígete hacia la playa de Las Teresitas para disfrutar de la costa.

Esta ruta te permitirá conocer los puntos más emblemáticos y agradables de Santa Cruz de Tenerife. 

Fuente: guía turística de Tenerife. ¿Quieres que te recomiende algún lugar para comer o más detalles sobre algún punto?

--- Turno 2 ---
Usuario: ¿Qué tiempo hará allí el 2026-06-20?


2026-06-16 20:49:24,067 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 20:49:24,085 - INFO - Intento de llamada a get_weather para la fecha: '2026-06-20' en 'Santa Cruz de Tenerife'
2026-06-16 20:49:24,098 - INFO - Llamada exitosa. Clima devuelto: soleado con una temperatura media de 24°C
2026-06-16 20:49:26,017 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Asistente: El 20 de junio de 2026 en Santa Cruz de Tenerife se espera un día soleado con una temperatura media de 24°C. ¿Quieres que te ayude con algo más para tu visita?

--- Turno 3 ---
Usuario: Genial. Siguiendo con la ruta que me dijiste antes, ¿dónde sugieres que aparque al principio?


2026-06-16 20:49:27,668 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 20:49:27,951 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 20:49:30,736 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Asistente: Te sugiero aparcar en el aparcamiento del Parque Marítimo César Manrique al inicio de la ruta en Santa Cruz de Tenerife. Este lugar es ideal porque está muy cerca de los principales puntos de interés que visitarás, como la Avenida Marítima, el Auditorio de Tenerife y la Plaza de España. Además, el Parque Marítimo es un conjunto de piscinas naturales que también puedes disfrutar.

Fuente: guía turística de Tenerife. ¿Quieres que te ayude con más detalles sobre el aparcamiento o la ruta?

Pruebas finalizadas. Revisa las citaciones del RAG y el log del clima.
